# trajopt-jax: the brachistochrone

The fastest path for a bead sliding under gravity from rest to a point 10 m across and 5 m down. We declare the model, solve on a coarse mesh, refine the mesh in passes, and compare with the exact cycloid.

Source: [`examples/brachistochrone.py`](https://github.com/thearn/trajopt-jax/blob/main/examples/brachistochrone.py)

## The problem

**States** $\mathbf{s} = (x, y, v, c)$: position (m), speed (m/s) and the cost $c$, here the elapsed time.
**Control** $\theta \in [0, \pi]$: the path's angle from the downward vertical.
**Parameter** $g = 9.81\ \mathrm{m/s^2}$.

**Dynamics**
$$
\dot x = v \sin\theta, \qquad \dot y = -v \cos\theta, \qquad \dot v = g \cos\theta, \qquad \dot c = 1 .
$$

**Objective**: the least time, the cost at the end,
$$
\min_{\theta(t),\, t_f} \; J = c(t_f) = t_f .
$$

**Boundary conditions**: from rest at the origin, to the point $(D, -H) = (10, -5)$ m within a tolerance $\epsilon$,
$$
\mathbf{s}(0) = (0, 0, 0, 0), \qquad |x(t_f) - D| \le \epsilon, \quad |y(t_f) + H| \le \epsilon, \qquad t_f \text{ free}.
$$

**Exact answer**: the cycloid $x = R(\phi - \sin\phi)$, $y = -R(1 - \cos\phi)$, through the end point, with descent time $t_f = \phi_f \sqrt{R / g}$.

## How trajopt-jax transcribes it

The flight is split into $N$ stages. In each stage the state is a cubic polynomial collocated at the three Radau IIA points (the dynamics hold exactly there), and the control is linear between knots. The result is a sparse nonlinear program in the knots and collocation states, solved by a primal-dual interior-point method whose linear algebra is a Riccati recursion over the stages. Mesh refinement then estimates each stage's error between its collocation points and splits the stages above tolerance.

In [ ]:
try:  # on Colab: install trajopt-jax
    import google.colab  # noqa: F401
    %pip install -q git+https://github.com/thearn/trajopt-jax matplotlib
    !wget -q https://raw.githubusercontent.com/thearn/trajopt-jax/main/examples/brachistochrone.py
except ImportError:
    import sys
    sys.path.insert(0, 'examples' if __import__('os').path.isdir('examples') else '.')

The model, the problem and the solve are in [`examples/brachistochrone.py`](https://github.com/thearn/trajopt-jax/blob/main/examples/brachistochrone.py) (a few dozen lines): the `Bead` class declares the states, control and dynamics above (`rhs`), the `Leg` with a `Point` end event is the boundary condition, and `main()` solves on 4 stages, refines, and returns the times and the path.

In [ ]:
import brachistochrone as b

out = b.main()

The refined path against the exact cycloid:

In [ ]:
import math

import matplotlib.pyplot as plt
import numpy as np

D, H = 10.0, 5.0
lo, hi = 0.1, 2 * math.pi - 0.1
for _ in range(100):
    mid = 0.5 * (lo + hi)
    lo, hi = (mid, hi) if (mid - math.sin(mid)) / (1 - math.cos(mid)) < D / H else (lo, mid)
R, th = H / (1 - math.cos(lo)), np.linspace(0, lo, 200)
plt.plot(R * (th - np.sin(th)), -R * (1 - np.cos(th)), 'k--', label='cycloid (exact)')
plt.plot(out['path'][:, 0], out['path'][:, 1], 'o-', ms=3, label=f"trajopt-jax, {out['stages']} stages")
plt.axis('equal'); plt.xlabel('x (m)'); plt.ylabel('y (m)'); plt.legend()
plt.title(f"{out['refined']:.4f} s vs cycloid {out['cycloid']:.4f} s");